In [1]:
# 重新开始 - 问题1求解
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

# 读取预处理后的数据
print("=== 读取预处理数据 ===")

df_land = pd.read_excel('preprocessed_land_data.xlsx')
df_crops = pd.read_excel('preprocessed_crops_data.xlsx')

print("地块数据:")
print(df_land.head())
print(f"\n地块总数: {len(df_land)}")
print(f"地块类型分布:")
print(df_land['地块类型'].value_counts())

print("\n作物数据:")
print(df_crops.head())
print(f"\n作物总数: {len(df_crops)}")
print(f"作物类型分布:")
print(df_crops['作物类型'].value_counts())

# 读取JSON数据
with open('compatibility_matrix.json', 'r', encoding='utf-8') as f:
    compatibility_matrix = json.load(f)

with open('expected_sales_2023.json', 'r', encoding='utf-8') as f:
    expected_sales_2023 = json.load(f)

with open('cost_price_data.json', 'r', encoding='utf-8') as f:
    cost_price_data = json.load(f)

print("\n=== 兼容性矩阵键名 ===")
for key in compatibility_matrix.keys():
    print(f"'{key}'")

print("\n=== 基础参数设置 ===")
YEARS = list(range(2024, 2031))  # 2024-2030年
SEASONS = ['单季', '第一季', '第二季']

print(f"年份范围: {YEARS}")
print(f"季节: {SEASONS}")

# 获取所有作物编号
all_crops = list(df_crops['作物编号'].unique())

# 获取豆类作物编号
bean_crops = df_crops[df_crops['作物类型'].str.contains('豆类', na=False)]['作物编号'].tolist()

print(f"\n总作物数: {len(all_crops)}")
print(f"豆类作物数: {len(bean_crops)}")
print(f"豆类作物编号: {bean_crops}")

# 建立地块-季节兼容性 - 使用正确的键名
land_season_compatibility = {
    '平旱地': ['单季'],
    '梯田': ['单季'],
    '山坡地': ['单季'],
    '水浇地': ['单季', '第一季', '第二季'],
    '普通大棚 ': ['第一季', '第二季'],
    '智慧大棚': ['第一季', '第二季']
}

# 特殊作物种植季次限制
special_crop_seasons = {
    '大白菜': ['第二季'],
    '白萝卜': ['第二季'],
    '红萝卜': ['第二季']
}

# 获取特殊作物编号
special_crop_ids = []
for crop_name in special_crop_seasons.keys():
    crop_match = df_crops[df_crops['作物名称'] == crop_name]
    if len(crop_match) > 0:
        crop_id = crop_match['作物编号'].iloc[0]
        special_crop_ids.append(crop_id)

# 获取食用菌类作物编号
mushroom_crops = df_crops[df_crops['作物类型'] == '食用菌']['作物编号'].tolist()
special_crop_ids.extend(mushroom_crops)

print(f"\n特殊作物编号: {special_crop_ids}")

# 建立优化模型的基础数据结构
print("\n=== 建立地块-季节-作物兼容性 ===")

land_season_crop_compatibility = {}

for land_name in df_land['地块名称']:
    land_type = df_land[df_land['地块名称'] == land_name]['地块类型'].iloc[0]
    land_season_crop_compatibility[land_name] = {}
    
    # 确保地块类型在兼容性矩阵中
    if land_type not in compatibility_matrix:
        print(f"警告: 地块类型 '{land_type}' 不在兼容性矩阵中")
        continue
        
    for season in land_season_compatibility[land_type]:
        compatible_crops = []
        
        # 根据地块类型获取兼容作物类型
        for crop_type in compatibility_matrix[land_type]:
            compatible_crops.extend(compatibility_matrix[land_type][crop_type])
        
        # 应用特殊季节限制
        if season == '第二季':
            # 第二季只能种植特殊作物
            compatible_crops = [crop for crop in compatible_crops if crop in special_crop_ids]
        else:
            # 第一季和单季不能种植特殊作物
            compatible_crops = [crop for crop in compatible_crops if crop not in special_crop_ids]
        
        land_season_crop_compatibility[land_name][season] = compatible_crops

print("地块-季节-作物兼容性建立完成！")

# 显示前5个地块的兼容性
print("\n=== 前5个地块兼容性分析 ===")
for land_name in list(land_season_crop_compatibility.keys())[:5]:
    print(f"\n{land_name}:")
    for season, crops in land_season_crop_compatibility[land_name].items():
        print(f"  {season}: {len(crops)}种作物")

# 建立优化模型的关键参数
print("\n=== 建立关键参数 ===")

# 地块面积（亩）
land_areas = {}
for land_name in df_land['地块名称']:
    area = df_land[df_land['地块名称'] == land_name]['地块面积/亩'].iloc[0]
    land_areas[land_name] = area

print(f"总耕地面积: {sum(land_areas.values()):.1f}亩")

# 预期销售量（斤）
expected_sales = {}
for crop_id_str, data in expected_sales_2023.items():
    crop_id = int(crop_id_str)
    expected_sales[crop_id] = data['expected_sales']

# 作物亩产量（斤/亩）
crop_yields = {}
for crop_id_str, data in cost_price_data.items():
    crop_id = int(crop_id_str)
    crop_yields[crop_id] = data['avg_yield']

# 作物种植成本（元/亩）
crop_costs = {}
for crop_id_str, data in cost_price_data.items():
    crop_id = int(crop_id_str)
    crop_costs[crop_id] = data['avg_cost']

# 作物销售价格（元/斤）
crop_prices = {}
for crop_id_str, data in cost_price_data.items():
    crop_id = int(crop_id_str)
    crop_prices[crop_id] = data['avg_price']

print("关键参数建立完成！")
print(f"地块数: {len(land_areas)}")
print(f"预期销售量记录数: {len(expected_sales)}")
print(f"作物亩产量记录数: {len(crop_yields)}")
print(f"作物成本记录数: {len(crop_costs)}")
print(f"作物价格记录数: {len(crop_prices)}")

=== 读取预处理数据 ===


地块数据:
  地块名称 地块类型  地块面积/亩                                                说明 
0   A1  平旱地    80.0  (1) 平旱地、梯田和山坡地每年都只能种植一季作物。\n\n(2) 水浇地每年可以种植一季也...
1   A2  平旱地    55.0                                                NaN
2   A3  平旱地    35.0                                                NaN
3   A4  平旱地    72.0                                                NaN
4   A5  平旱地    68.0                                                NaN

地块总数: 54
地块类型分布:
地块类型
普通大棚     16
梯田       14
水浇地       8
平旱地       6
山坡地       6
智慧大棚      4
Name: count, dtype: int64

作物数据:
   作物编号 作物名称    作物类型                种植耕地  \
0     1   黄豆  粮食（豆类）  平旱地\n\n梯田\n\n山坡地\n   
1     2   黑豆  粮食（豆类）                 NaN   
2     3   红豆  粮食（豆类）                 NaN   
3     4   绿豆  粮食（豆类）                 NaN   
4     5   爬豆  粮食（豆类）                 NaN   

                                                  说明  
0  (1) 平旱地、梯田和山坡地每年适宜单季种植粮食类作物（水稻除外）。\n\n(2) 水浇地每...  
1                                                NaN  
2         

In [2]:
# 修复普通大棚的空格问题
print("=== 修复普通大棚空格问题 ===")

# 更新地块类型名称 - 去掉空格
df_land['地块类型'] = df_land['地块类型'].str.strip()

print("更新后的地块类型分布:")
print(df_land['地块类型'].value_counts())

# 更新地块-季节兼容性定义
land_season_compatibility = {
    '平旱地': ['单季'],
    '梯田': ['单季'],
    '山坡地': ['单季'],
    '水浇地': ['单季', '第一季', '第二季'],
    '普通大棚': ['第一季', '第二季'],
    '智慧大棚': ['第一季', '第二季']
}

# 重新建立地块-季节-作物兼容性
land_season_crop_compatibility = {}

for land_name in df_land['地块名称']:
    land_type = df_land[df_land['地块名称'] == land_name]['地块类型'].iloc[0]
    land_season_crop_compatibility[land_name] = {}
    
    # 确保地块类型在兼容性矩阵中
    if land_type not in compatibility_matrix:
        print(f"警告: 地块类型 '{land_type}' 不在兼容性矩阵中")
        continue
        
    for season in land_season_compatibility[land_type]:
        compatible_crops = []
        
        # 根据地块类型获取兼容作物类型
        for crop_type in compatibility_matrix[land_type]:
            compatible_crops.extend(compatibility_matrix[land_type][crop_type])
        
        # 应用特殊季节限制
        if season == '第二季':
            # 第二季只能种植特殊作物
            compatible_crops = [crop for crop in compatible_crops if crop in special_crop_ids]
        else:
            # 第一季和单季不能种植特殊作物
            compatible_crops = [crop for crop in compatible_crops if crop not in special_crop_ids]
        
        land_season_crop_compatibility[land_name][season] = compatible_crops

print("\n=== 更新后的地块-季节-作物兼容性 ===")

# 显示所有地块类型的兼容性概况
print("\n各地块类型的兼容性概况:")
for land_type in df_land['地块类型'].unique():
    if land_type in land_season_compatibility:
        seasons = land_season_compatibility[land_type]
        print(f"\n{land_type}:")
        for season in seasons:
            # 计算该地块类型在该季节的可种植作物数量
            total_compatible = 0
            for land_name in df_land[df_land['地块类型'] == land_type]['地块名称']:
                if land_name in land_season_crop_compatibility and season in land_season_crop_compatibility[land_name]:
                    total_compatible += len(land_season_crop_compatibility[land_name][season])
            
            avg_compatible = total_compatible / len(df_land[df_land['地块类型'] == land_type])
            print(f"  {season}: 平均{avg_compatible:.1f}种作物")

# 建立优化模型的关键参数
print("\n=== 建立优化模型的关键参数 ===")

# 地块面积（亩） - 更新后的数据
land_areas = {}
for land_name in df_land['地块名称']:
    area = df_land[df_land['地块名称'] == land_name]['地块面积/亩'].iloc[0]
    land_areas[land_name] = area

print(f"总耕地面积: {sum(land_areas.values()):.1f}亩")

# 计算各作物的利润率
print("\n=== 作物利润率分析 ===")

crop_profits = {}
for crop_id in all_crops:
    if crop_id in crop_yields and crop_id in crop_prices and crop_id in crop_costs:
        profit_per_mu = crop_yields[crop_id] * crop_prices[crop_id] - crop_costs[crop_id]
        crop_profits[crop_id] = profit_per_mu

# 按利润率排序
sorted_profits = sorted(crop_profits.items(), key=lambda x: x[1], reverse=True)

print("利润率最高的前10种作物:")
for i, (crop_id, profit) in enumerate(sorted_profits[:10]):
    crop_name = df_crops[df_crops['作物编号'] == crop_id]['作物名称'].iloc[0]
    print(f"{i+1}. {crop_name} (编号{crop_id}): 每亩利润{profit:.0f}元")

print("\n利润率最低的前10种作物:")
for i, (crop_id, profit) in enumerate(sorted_profits[-10:]):
    crop_name = df_crops[df_crops['作物编号'] == crop_id]['作物名称'].iloc[0]
    print(f"{i+1}. {crop_name} (编号{crop_id}): 每亩利润{profit:.0f}元")

print("\n=== 优化模型基础建立完成 ===")

=== 修复普通大棚空格问题 ===
更新后的地块类型分布:
地块类型
普通大棚    16
梯田      14
水浇地      8
平旱地      6
山坡地      6
智慧大棚     4
Name: count, dtype: int64

=== 更新后的地块-季节-作物兼容性 ===

各地块类型的兼容性概况:

平旱地:
  单季: 平均16.0种作物

梯田:
  单季: 平均16.0种作物

山坡地:
  单季: 平均16.0种作物

水浇地:
  单季: 平均29.0种作物
  第一季: 平均29.0种作物
  第二季: 平均3.0种作物

普通大棚:
  第一季: 平均18.0种作物
  第二季: 平均7.0种作物

智慧大棚:
  第一季: 平均18.0种作物
  第二季: 平均3.0种作物

=== 建立优化模型的关键参数 ===
总耕地面积: 1213.0亩

=== 作物利润率分析 ===
利润率最高的前10种作物:
1. 榆黄菇 (编号38): 每亩利润284500元
2. 白灵菇 (编号40): 每亩利润150000元
3. 黄瓜 (编号29): 每亩利润97383元
4. 羊肚菌 (编号41): 每亩利润90000元
5. 香菇 (编号39): 每亩利润74000元
6. 空心菜 (编号32): 每亩利润47933元
7. 茄子 (编号22): 每亩利润39893元
8. 豇豆 (编号17): 每亩利润25529元
9. 包菜 (编号26): 每亩利润25010元
10. 芹菜 (编号34): 每亩利润24676元

利润率最低的前10种作物:
1. 水稻 (编号16): 每亩利润2820元
2. 红豆 (编号3): 每亩利润2785元
3. 玉米 (编号7): 每亩利润2350元
4. 爬豆 (编号5): 每亩利润2316元
5. 小麦 (编号6): 每亩利润2210元
6. 谷子 (编号8): 每亩利润2205元
7. 绿豆 (编号4): 每亩利润1972元
8. 莜麦 (编号14): 每亩利润1800元
9. 大麦 (编号15): 每亩利润1400元
10. 黄豆 (编号1): 每亩利润835元

=== 优化模型基础建立完成 ===


In [3]:
# 建立问题1的优化模型
print("=== 问题1求解 - 确定性优化模型 ===")

# 建立优化模型的基础结构
class CropOptimizationModel:
    def __init__(self, years, seasons, land_data, crop_data, compatibility_data, 
                 expected_sales, crop_yields, crop_costs, crop_prices):
        self.years = years
        self.seasons = seasons
        self.land_data = land_data
        self.crop_data = crop_data
        self.compatibility_data = compatibility_data
        self.expected_sales = expected_sales
        self.crop_yields = crop_yields
        self.crop_costs = crop_costs
        self.crop_prices = crop_prices
        
        # 获取豆类作物
        self.bean_crops = crop_data[crop_data['作物类型'].str.contains('豆类', na=False)]['作物编号'].tolist()
        
        # 初始化种植方案
        self.planting_plans = {}
        
    def calculate_profit(self, crop_id, area, scenario='waste'):
        """计算某作物在某面积下的利润"""
        if crop_id not in self.crop_yields or crop_id not in self.crop_prices or crop_id not in self.crop_costs:
            return 0
            
        yield_per_mu = self.crop_yields[crop_id]
        price = self.crop_prices[crop_id]
        cost_per_mu = self.crop_costs[crop_id]
        
        total_yield = area * yield_per_mu
        
        if crop_id in self.expected_sales:
            expected_sales = self.expected_sales[crop_id]
            
            if total_yield <= expected_sales:
                # 产量不超过预期销售量
                revenue = total_yield * price
            else:
                # 产量超过预期销售量
                if scenario == 'waste':
                    # 情况(1): 超过部分浪费
                    revenue = expected_sales * price
                else:
                    # 情况(2): 超过部分降价50%出售
                    revenue = expected_sales * price + (total_yield - expected_sales) * price * 0.5
        else:
            # 没有预期销售量数据，按完全销售处理
            revenue = total_yield * price
        
        total_cost = area * cost_per_mu
        profit = revenue - total_cost
        
        return profit
    
    def optimize_single_year(self, year, scenario='waste'):
        """优化单年的种植方案"""
        print(f"\n优化{year}年种植方案...")
        
        # 初始化种植记录
        planting_records = []
        
        # 按地块进行优化
        for land_name in self.land_data['地块名称']:
            land_type = self.land_data[self.land_data['地块名称'] == land_name]['地块类型'].iloc[0]
            land_area = self.land_data[self.land_data['地块名称'] == land_name]['地块面积/亩'].iloc[0]
            
            # 获取该地块可种植的季节
            if land_name not in self.compatibility_data:
                continue
                
            for season in self.compatibility_data[land_name]:
                compatible_crops = self.compatibility_data[land_name][season]
                
                if not compatible_crops:
                    continue
                
                # 按利润率排序选择作物
                crop_profits = []
                for crop_id in compatible_crops:
                    profit = self.calculate_profit(crop_id, land_area, scenario)
                    crop_profits.append((crop_id, profit))
                
                # 选择利润最高的作物
                if crop_profits:
                    best_crop = max(crop_profits, key=lambda x: x[1])
                    crop_id, profit = best_crop
                    
                    # 记录种植方案
                    crop_name = self.crop_data[self.crop_data['作物编号'] == crop_id]['作物名称'].iloc[0]
                    
                    planting_records.append({
                        '年份': year,
                        '地块名称': land_name,
                        '地块类型': land_type,
                        '种植季次': season,
                        '作物编号': crop_id,
                        '作物名称': crop_name,
                        '种植面积/亩': land_area,
                        '预计利润/元': profit
                    })
        
        return planting_records
    
    def optimize_multi_year(self, scenario='waste'):
        """优化多年的种植方案"""
        print(f"=== 开始优化{len(self.years)}年种植方案 ===")
        print(f"处理情况: {scenario}")
        
        all_planting_records = []
        
        for year in self.years:
            year_records = self.optimize_single_year(year, scenario)
            all_planting_records.extend(year_records)
            
            # 计算年度总利润
            total_profit = sum(record['预计利润/元'] for record in year_records)
            total_area = sum(record['种植面积/亩'] for record in year_records)
            
            print(f"{year}年: 总种植面积{total_area:.1f}亩, 预计总利润{total_profit:.0f}元")
        
        return all_planting_records

# 创建优化模型实例
optimization_model = CropOptimizationModel(
    years=YEARS,
    seasons=SEASONS,
    land_data=df_land,
    crop_data=df_crops,
    compatibility_data=land_season_crop_compatibility,
    expected_sales=expected_sales,
    crop_yields=crop_yields,
    crop_costs=crop_costs,
    crop_prices=crop_prices
)

print("优化模型初始化完成！")

# 求解情况(1): 超过部分浪费
print("\n=== 情况(1): 超过部分浪费 ===")
scenario1_results = optimization_model.optimize_multi_year(scenario='waste')

# 求解情况(2): 超过部分降价50%出售
print("\n=== 情况(2): 超过部分降价50%出售 ===")
scenario2_results = optimization_model.optimize_multi_year(scenario='discount')

print("\n=== 优化完成 ===")
print(f"情况(1)记录数: {len(scenario1_results)}")
print(f"情况(2)记录数: {len(scenario2_results)}")

# 转换为DataFrame格式
scenario1_df = pd.DataFrame(scenario1_results)
scenario2_df = pd.DataFrame(scenario2_results)

print("\n情况(1)前5条记录:")
print(scenario1_df.head())

print("\n情况(2)前5条记录:")
print(scenario2_df.head())

=== 问题1求解 - 确定性优化模型 ===
优化模型初始化完成！

=== 情况(1): 超过部分浪费 ===
=== 开始优化7年种植方案 ===
处理情况: waste

优化2024年种植方案...
2024年: 总种植面积1443.0亩, 预计总利润13956286元

优化2025年种植方案...
2025年: 总种植面积1443.0亩, 预计总利润13956286元

优化2026年种植方案...
2026年: 总种植面积1443.0亩, 预计总利润13956286元

优化2027年种植方案...
2027年: 总种植面积1443.0亩, 预计总利润13956286元

优化2028年种植方案...
2028年: 总种植面积1443.0亩, 预计总利润13956286元

优化2029年种植方案...


2029年: 总种植面积1443.0亩, 预计总利润13956286元

优化2030年种植方案...


2030年: 总种植面积1443.0亩, 预计总利润13956286元

=== 情况(2): 超过部分降价50%出售 ===
=== 开始优化7年种植方案 ===
处理情况: discount

优化2024年种植方案...


2024年: 总种植面积1443.0亩, 预计总利润19244794元

优化2025年种植方案...
2025年: 总种植面积1443.0亩, 预计总利润19244794元

优化2026年种植方案...
2026年: 总种植面积1443.0亩, 预计总利润19244794元

优化2027年种植方案...
2027年: 总种植面积1443.0亩, 预计总利润19244794元

优化2028年种植方案...
2028年: 总种植面积1443.0亩, 预计总利润19244794元

优化2029年种植方案...
2029年: 总种植面积1443.0亩, 预计总利润19244794元

优化2030年种植方案...
2030年: 总种植面积1443.0亩, 预计总利润19244794元

=== 优化完成 ===
情况(1)记录数: 630
情况(2)记录数: 630



情况(1)前5条记录:
     年份 地块名称 地块类型 种植季次  作物编号 作物名称  种植面积/亩    预计利润/元
0  2024   A1  平旱地   单季     7   玉米    80.0  188000.0
1  2024   A2  平旱地   单季     9   高粱    55.0  158000.0
2  2024   A3  平旱地   单季     9   高粱    35.0  112000.0
3  2024   A4  平旱地   单季     7   玉米    72.0  169200.0
4  2024   A5  平旱地   单季     3   红豆    68.0  164300.0

情况(2)前5条记录:
     年份 地块名称 地块类型 种植季次  作物编号 作物名称  种植面积/亩    预计利润/元
0  2024   A1  平旱地   单季     9   高粱    80.0  202000.0
1  2024   A2  平旱地   单季     9   高粱    55.0  167000.0
2  2024   A3  平旱地   单季     9   高粱    35.0  112000.0
3  2024   A4  平旱地   单季     9   高粱    72.0  190800.0
4  2024   A5  平旱地   单季     9   高粱    68.0  185200.0


In [4]:
# 将结果保存为Excel格式
print("=== 将结果保存为Excel格式 ===")

# 按照模板要求格式化结果
# 模板包含: 年份, 地块名称, 作物编号, 作物名称, 种植面积/亩, 种植季次

def format_results_for_excel(results_df):
    """格式化结果为Excel格式"""
    formatted_df = results_df[['年份', '地块名称', '作物编号', 
                              '作物名称', '种植面积/亩', '种植季次']].copy()
    
    # 确保列顺序
    formatted_df = formatted_df[['年份', '地块名称', '作物编号', 
                                '作物名称', '种植面积/亩', '种植季次']]
    
    # 按年份和地块排序
    formatted_df = formatted_df.sort_values(['年份', '地块名称'])
    
    return formatted_df

# 格式化结果
scenario1_formatted = format_results_for_excel(scenario1_df)
scenario2_formatted = format_results_for_excel(scenario2_df)

print("情况(1)格式化后的数据:")
print(scenario1_formatted.head(10))

print(f"\n情况(1)总记录数: {len(scenario1_formatted)}")
print(f"情况(2)总记录数: {len(scenario2_formatted)}")

# 保存结果到Excel文件
print("\n=== 保存结果到Excel文件 ===")

# 保存情况(1)结果
scenario1_formatted.to_excel('result1_1.xlsx', index=False)
print("情况(1)结果已保存为 result1_1.xlsx")

# 保存情况(2)结果
scenario2_formatted.to_excel('result1_2.xlsx', index=False)
print("情况(2)结果已保存为 result1_2.xlsx")

# 进行结果分析
print("\n=== 结果分析 ===")

# 分析各年的种植面积和利润
print("\n情况(1) - 各年种植面积和利润:")
for year in YEARS:
    year_data = scenario1_df[scenario1_df['年份'] == year]
    total_area = year_data['种植面积/亩'].sum()
    total_profit = year_data['预计利润/元'].sum()
    print(f"{year}年: 种植面积{total_area:.1f}亩, 预计利润{total_profit:.0f}元")

print("\n情况(2) - 各年种植面积和利润:")
for year in YEARS:
    year_data = scenario2_df[scenario2_df['年份'] == year]
    total_area = year_data['种植面积/亩'].sum()
    total_profit = year_data['预计利润/元'].sum()
    print(f"{year}年: 种植面积{total_area:.1f}亩, 预计利润{total_profit:.0f}元")

# 分析作物种类分布
print("\n=== 作物种类分布分析 ===")

print("\n情况(1) - 各作物种植面积:")
crop_area_scenario1 = scenario1_df.groupby('作物名称')['种植面积/亩'].sum().sort_values(ascending=False)
print(crop_area_scenario1.head(10))

print("\n情况(2) - 各作物种植面积:")
crop_area_scenario2 = scenario2_df.groupby('作物名称')['种植面积/亩'].sum().sort_values(ascending=False)
print(crop_area_scenario2.head(10))

print("\n=== 问题1求解完成 ===")
print("两种情况的最优种植方案已生成并保存")

=== 将结果保存为Excel格式 ===
情况(1)格式化后的数据:
      年份 地块名称  作物编号 作物名称  种植面积/亩 种植季次
0   2024   A1     7   玉米    80.0   单季
1   2024   A2     9   高粱    55.0   单季
2   2024   A3     9   高粱    35.0   单季
3   2024   A4     7   玉米    72.0   单季
4   2024   A5     3   红豆    68.0   单季
5   2024   A6     9   高粱    55.0   单季
6   2024   B1     3   红豆    60.0   单季
15  2024  B10    10   黍子    25.0   单季
16  2024  B11     3   红豆    60.0   单季
17  2024  B12     9   高粱    45.0   单季

情况(1)总记录数: 630
情况(2)总记录数: 630

=== 保存结果到Excel文件 ===
情况(1)结果已保存为 result1_1.xlsx


情况(2)结果已保存为 result1_2.xlsx

=== 结果分析 ===

情况(1) - 各年种植面积和利润:
2024年: 种植面积1443.0亩, 预计利润13956286元
2025年: 种植面积1443.0亩, 预计利润13956286元
2026年: 种植面积1443.0亩, 预计利润13956286元
2027年: 种植面积1443.0亩, 预计利润13956286元
2028年: 种植面积1443.0亩, 预计利润13956286元
2029年: 种植面积1443.0亩, 预计利润13956286元
2030年: 种植面积1443.0亩, 预计利润13956286元

情况(2) - 各年种植面积和利润:
2024年: 种植面积1443.0亩, 预计利润19244794元
2025年: 种植面积1443.0亩, 预计利润19244794元
2026年: 种植面积1443.0亩, 预计利润19244794元
2027年: 种植面积1443.0亩, 预计利润19244794元
2028年: 种植面积1443.0亩, 预计利润19244794元
2029年: 种植面积1443.0亩, 预计利润19244794元
2030年: 种植面积1443.0亩, 预计利润19244794元

=== 作物种类分布分析 ===

情况(1) - 各作物种植面积:
作物名称
高粱     3605.0
玉米     1666.0
红豆     1316.0
芹菜      994.0
大白菜     779.8
红薯      707.0
茄子      364.0
黍子      350.0
豇豆      168.0
黄瓜       84.0
Name: 种植面积/亩, dtype: float64

情况(2) - 各作物种植面积:
作物名称
高粱     6202.0
黄瓜     1610.0
红薯     1442.0
大白菜     779.8
榆黄菇      67.2
Name: 种植面积/亩, dtype: float64

=== 问题1求解完成 ===
两种情况的最优种植方案已生成并保存
